# 10. Phase 5 원문 대조와 요약 누락 확인 — 추가 API 호출 없음

유영관 · 2026-11-11

업로드한 연말정산 위탁운영 용역 요약은 **18개 청크 처리·기존 인용 20개 대조에 통과**했습니다.
그러나 입력을 모두 보냈다는 사실만으로 요약 내용이 완전해지지는 않습니다.
원문에 있는데 기존 요약에서 빠진 정산 조건·기간 이후 지원·선정방식을 확인했습니다.

이 노트북은 원문 추가 확인 검사를 설치하고, 기존 `result.json`에 대해 **모델을 다시 호출하지 않고** 원문을 대조합니다.
생성 답변을 자동으로 고치거나, 사람 검토를 통과한 것으로 표시하지 않습니다.
원문·청크·인덱스·캐시·기존 결과·API 실행 옵션은 보존합니다.

기존 Phase 5의 `notebooks/` 폴더에 이 파일을 넣고, 성공했던 커널에서 Run All 하세요.
현재 요약을 확인하기 위해 08번을 다시 실행할 필요는 없습니다.


## 이번 파일의 11개 항목 검토

아래는 제공한 실행 결과와 추출 원문을 대조한 사례 검토입니다. 전체 문서 집합의 정답률 평가는 아닙니다.

| 항목 | 검토 결과 |
| --- | --- |
| 사업명 | 원문과 일치 |
| 발주기관 | 본문에 기관명이 등장하지만 발주 역할 근거를 추가로 확인해야 함. 파일명만으로 확정하지 않음 |
| 목적 | 두 가지 목적이 원문에 연결됨 |
| 주요 범위·필수요건 | 원문 업무 범위가 정리됨. 긴 문장은 후속 검토 시 세부 요건으로 나눌 수 있음 |
| 예산·VAT | 57,246,000원·부가가치세 제외는 일치. 단가계약·실제 대상인원 정산 조건을 추가 확인 |
| 수행기간 | 계약체결일~2027.3.17은 일치. 과업기간 이후 후속 지원 조건을 추가 확인 |
| 참가·수행자격 | 세무사·경력·상주인력 요건은 일치. 이는 수행인력 요건이며 입찰 참가자격 전체를 뜻하지 않음 |
| 입찰 제출서류 | 현재 추출 원문에서 입찰 제출서류를 확정하지 않음. 착수·준공 제출서류와 구별 |
| 제출기한 | 입찰 마감일을 확정하지 않음. 착수계 제출 기한을 입찰 마감으로 사용하지 않음 |
| 제출방식 | 현재 자료로 입찰 제출방식을 확정하지 않음 |
| 평가기준 | 적격심사(제한적최저가) 낙찰제는 원문에 있음. 세부 평가항목·배점은 별도 확인 |

발주기관·입찰 조건을 확인하려면 별도 공고문 등 추가 자료가 필요할 수 있습니다. 현재 파일에 공고문 참조 지시가 없는데 임의로 그런 지시를 만들지는 않습니다.


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = None
RESULT_PATH = None  # 기본: phase5_outputs/runs 아래 가장 최근 result.json
if PROJECT_ROOT is None:
    candidates = [Path.cwd(), Path.cwd().parent,
                  Path.cwd() / 'rfp_phase5_local_20261111',
                  Path.cwd() / 'output/rfp_phase5_local_20261111']
    ROOT = next((p.resolve() for p in candidates if (p / 'rfp_phase5/runtime.py').is_file()
                 and (p / 'index/index_meta.json').is_file()), None)
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
if ROOT is None:
    raise FileNotFoundError('PROJECT_ROOT에 기존 Phase 5 프로젝트 폴더를 지정하세요.')
print('Python:', sys.executable)
print('프로젝트:', ROOT)
print('API 호출: 0회 / 생성 재실행: 없음')


Python: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase3_local_20261111/.venv/bin/python
프로젝트: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase5_local_20261111
API 호출: 0회 / 생성 재실행: 없음


## 1. 원문 추가 확인 검사 적용

기존 코드를 백업하고 버전이 일치하는 파일만 수정합니다. 앞으로 08번에서 생성할 때에도 정해진 원문 규칙의 추가 확인 후보를 표시합니다.
새 검사가 모든 누락을 찾는 것은 아닙니다. 후보는 기존 답변에 자동으로 합치지 않습니다.
08번에서 새 코드를 사용할 때는 그 노트북의 커널을 재시작하세요.


In [2]:
"""Guarded local source-review upgrade. Never imports a model client or calls APIs."""
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import shutil

PATCH_ID = 'phase5-source-audit-v1'
ALLOWED = {'rfp_phase5/source_audit.py', 'rfp_phase5/summary.py', 'tests/test_phase5_source_audit.py'}


def sha(value):
    return hashlib.sha256(value).hexdigest()


def apply_source_audit(project, manifest, embedded_files):
    project = Path(project).expanduser().resolve()
    if manifest.get('patch_id') != PATCH_ID or set(manifest.get('files', {})) != ALLOWED:
        raise ValueError('원문 대조 수정 파일 목록 또는 버전이 올바르지 않습니다.')
    if not (project / 'rfp_phase5/runtime.py').is_file() or not (project / 'index/index_meta.json').is_file():
        raise ValueError('기존 Phase 5 프로젝트의 최상위 폴더를 지정하세요.')
    staged, already = {}, []
    # Validate every existing file before the first write. Custom local edits
    # are rejected; never replace a file whose version is unknown.
    for relative, spec in manifest['files'].items():
        target = (project / relative).resolve()
        target.relative_to(project)
        new = embedded_files[relative].encode('utf-8')
        if sha(new) != spec['new_sha256']:
            raise ValueError('수정 내용의 해시가 다릅니다: ' + relative)
        current = sha(target.read_bytes()) if target.is_file() else None
        if current == spec['new_sha256']:
            already.append(relative)
        elif current != spec['old_sha256']:
            raise ValueError('직접 수정했거나 다른 버전인 파일입니다. 덮어쓰지 않습니다: ' + relative)
        else:
            staged[relative] = new
    backup = None
    written = []
    try:
        if staged:
            backup = project / 'phase5_outputs/patch_backups' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
            backup.mkdir(parents=True)
            for relative, content in staged.items():
                target = project / relative
                if target.is_file():
                    old = backup / relative
                    old.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(target, old)
                target.parent.mkdir(parents=True, exist_ok=True)
                temporary = target.with_name(target.name + '.source-audit-tmp')
                temporary.write_bytes(content)
                temporary.replace(target)
                written.append(relative)
    except Exception:
        for relative in reversed(written):
            old = backup / relative
            if old.is_file():
                shutil.copy2(old, project / relative)
            else:
                (project / relative).unlink(missing_ok=True)
        raise
    report = {'patch_id': PATCH_ID, 'modified_files': list(staged),
              'already_applied_files': already,
              'backup_directory': str(backup) if backup else None,
              'api_calls': 0, 'data_or_index_modified': False, 'cache_modified': False,
              'config_or_notebook_flags_modified': False, 'retry_executed': False,
              'restart_08_notebook_kernel_before_future_generation': True,
              'original_answers_modified': False, 'semantic_quality_verified': False}
    if backup:
        (backup / 'patch_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n')
    return report

MANIFEST = json.loads('{"patch_id": "phase5-source-audit-v1", "files": {"rfp_phase5/source_audit.py": {"old_sha256": null, "new_sha256": "78fa5faaa7e61028d7cd79996cb37caa0e4dd9e2a4c38a8e2db3103204cf285a"}, "rfp_phase5/summary.py": {"old_sha256": "85618cc2c90a82e581ccb48c563f33fbbf4d155f7c57e535953f57c70859a970", "new_sha256": "7aeaa4869c3ed95ada3874af2b3e20decdfe70f30ea2e1d9aa25e16a6323c8c1"}, "tests/test_phase5_source_audit.py": {"old_sha256": null, "new_sha256": "9d94189770c5586e2b61d8a6140a129d561588e672fee51e7267aaaada4ae2a9"}}}')
EMBEDDED_FILES = json.loads('{"rfp_phase5/source_audit.py": "\\"\\"\\"Source-backed omission hints over processed input, without generation.\\n\\nQuote provenance does not establish completeness. These limited surface rules\\nfind source lines absent from the corresponding field\'s text/quotes. They do\\nnot establish semantic error, resolve conflicts, or fill missing answer fields.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport html\\nimport json\\nimport re\\n\\nfrom .fields import FIELD_BY_ID, parse_field_claim\\n\\n\\nRULES = (\\n    (\'unit_price_settlement\', \'budget_vat\', r\'단가\\\\s*계약\',\\n     (\'단가계약\',), \'단가계약과 실제 대상·물량에 따른 정산 조건을 확인하세요.\'),\\n    (\'post_period_support\', \'duration\', r\'(?:과업|용역|사업|계약)\\\\s*기간\\\\s*(?:이후|종료\\\\s*후)\',\\n     (\'기간이후\', \'기간종료후\'), \'계약상 기간이 끝난 뒤에도 남는 지원 의무가 있는지 확인하세요.\'),\\n    (\'selection_method\', \'evaluation\', r\'(?:낙찰자\\\\s*선정\\\\s*방식|낙찰자\\\\s*결정\\\\s*방법)\\\\s*[:：]\\\\s*\\\\S\',\\n     (\'선정방식\', \'결정방법\', \'적격심사\', \'협상에의한계약\'),\\n     \'선정방식과 세부 평가항목·배점을 구분하세요. 선정방식만으로 배점표를 추정하지 않습니다.\'),\\n)\\n\\n\\ndef _compact(value):\\n    return \'\'.join(str(value).split())\\n\\n\\ndef _digest(value):\\n    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True,\\n        separators=(\',\', \':\'), allow_nan=False).encode(\'utf-8\')).hexdigest()\\n\\n\\ndef audit_summary_sources(index, registry, result):\\n    \\"\\"\\"Revalidate original claims, then inspect only truly processed source chunks.\\n\\n    ``fields`` is a display projection and is never trusted as the source of\\n    claims or facts. Original generated claims and quotes remain unchanged.\\n    \\"\\"\\"\\n    did = result.get(\'doc_id\')\\n    canonical = list(index.resolve_doc_ids([did]))\\n    if len(canonical) != 1 or canonical[0] != did:\\n        raise ValueError(\'대표 문서 ID 한 개의 요약 결과가 필요합니다.\')\\n    full = list(registry.validate_chunks(index.get_document_chunks(did), allowed_doc_ids=[did]))\\n    by_id = {chunk[\'chunk_id\']: chunk for chunk in full}\\n    full_ids = [chunk[\'chunk_id\'] for chunk in full]\\n    declared = result.get(\'source_chunk_ids\') or []\\n    processed = result.get(\'processed_chunk_ids\') or []\\n    if (declared != full_ids or len(processed) != len(set(processed))\\n            or any(cid not in by_id for cid in processed)\\n            or [cid for cid in full_ids if cid in set(processed)] != processed):\\n        raise ValueError(\'요약의 원문 청크 범위·순서 또는 처리 목록이 현재 문서와 다릅니다.\')\\n    chunks = [by_id[cid] for cid in processed]\\n    claims = result.get(\'claims\') or []\\n    by_field = {key: [] for key in FIELD_BY_ID}\\n    payload_claims = []\\n    for claim in claims:\\n        parsed = parse_field_claim(claim)\\n        by_field[parsed[\'field_id\']].append(claim)\\n        payload_claims.append({key: claim[key] for key in (\'doc_id\', \'category\', \'text\')} | {\\n            \'citations\': [{key: cite[key] for key in (\'chunk_id\', \'quote\')}\\n                          for cite in claim.get(\'citations\') or []]})\\n    validation = None\\n    if payload_claims:\\n        payload = {\'status\': result.get(\'status\'), \'claims\': payload_claims,\\n                   \'missing_information\': result.get(\'missing_information\') or []}\\n        validation = registry.validate_answer(payload, chunks, allowed_doc_ids=[did])\\n        if not validation[\'valid\']:\\n            raise ValueError(\'저장된 요약의 원문 인용을 다시 검증하지 못했습니다.\')\\n    elif processed:\\n        # A completed empty-evidence map can be audited, but it cannot receive\\n        # a provenance-valid answer verdict without any claims.\\n        validation = {\'valid\': False, \'claims\': []}\\n    candidates, seen = [], set()\\n    for rule_id, field_id, expression, markers, explanation in RULES:\\n        related = by_field[field_id]\\n        field_text = _compact(\'\\\\n\'.join(c[\'text\'] + \'\\\\n\' + \'\\\\n\'.join(\\n            cite[\'quote\'] for cite in c.get(\'citations\') or []) for c in related))\\n        # Surface presence only. A paraphrase may still be flagged; never turn\\n        # this decision into a semantic-error label or an automatic correction.\\n        if any(marker in field_text for marker in markers):\\n            continue\\n        for chunk in chunks:\\n            for quote in chunk[\'text\'].splitlines():\\n                if not quote.strip() or not re.search(expression, quote):\\n                    continue\\n                key = rule_id, did, quote\\n                if key in seen:\\n                    continue\\n                seen.add(key)\\n                candidate_claim = {\'doc_id\': did, \'category\': FIELD_BY_ID[field_id][\'category\'],\\n                    \'text\': quote, \'citations\': [{\'chunk_id\': chunk[\'chunk_id\'], \'quote\': quote}]}\\n                verified = registry.validate_answer({\'status\': \'answered\', \'claims\': [candidate_claim],\\n                    \'missing_information\': []}, [chunk], allowed_doc_ids=[did])\\n                if not verified[\'valid\']:\\n                    raise ValueError(\'원문 추가 확인 후보의 인용 검증에 실패했습니다.\')\\n                candidates.append({\'rule_id\': rule_id, \'field_id\': field_id,\\n                    \'label\': FIELD_BY_ID[field_id][\'label\'], \'source_quote\': quote,\\n                    \'citations\': verified[\'claims\'][0][\'citations\'],\\n                    \'explanation\': explanation, \'requires_review\': True,\\n                    \'automatic_semantic_error\': False, \'automatically_added_to_answer\': False})\\n    return {\'version\': \'phase5-source-audit-v1\', \'doc_id\': did,\\n            \'source_sha256\': registry.source_hashes[did],\\n            \'input_result_fingerprint\': _digest({\'doc_id\': did, \'claims\': claims,\\n                \'source_chunk_ids\': declared, \'processed_chunk_ids\': processed}),\\n            \'processed_chunk_count\': len(chunks), \'document_chunk_count\': len(full),\\n            \'all_input_chunks_processed\': processed == full_ids,\\n            \'original_claim_count\': len(claims),\\n            \'original_citation_count\': sum(len(c.get(\'citations\') or []) for c in claims),\\n            \'original_quotes_revalidated\': bool(validation and validation[\'valid\']),\\n            \'candidate_count\': len(candidates), \'candidates\': candidates,\\n            \'semantic_accuracy\': None, \'semantic_completeness_verified\': False,\\n            \'human_review_completed\': False, \'api_calls\': 0,\\n            \'scope\': \'이미 처리된 원문의 일부 표면 규칙을 대조한 추가 확인 후보. 모든 누락 탐지나 의미적 정답 판정은 아닙니다.\'}\\n\\n\\ndef render_source_audit(audit):\\n    def escape(value):\\n        text = html.escape(str(value), quote=False)\\n        text = re.sub(r\'([\\\\\\\\`*_{}\\\\[\\\\]()#!+\\\\-.])\', r\'\\\\\\\\\\\\1\', text)\\n        return text.replace(\'|\', \'&#124;\').replace(\'\\\\n\', \' / \')\\n    lines = [\'원문 추가 확인\', \'\',\\n        \'이미 전달된 원문에서 찾은 내용입니다. 기존 생성 답변에 자동으로 추가하지 않았습니다.\', \'\',\\n        \'| 항목 | 원문에서 찾은 내용 | 확인할 점 |\', \'| --- | --- | --- |\']\\n    for candidate in audit.get(\'candidates\') or []:\\n        lines.append(\'| \' + \' | \'.join(escape(candidate[key]) for key in\\n                     (\'label\', \'source_quote\', \'explanation\')) + \' |\')\\n    if not audit.get(\'candidates\'):\\n        lines.append(\'| 추가 후보 없음 | 지정된 규칙에서 발견하지 못함 | 요약 완전성을 뜻하지 않음 |\')\\n    if audit.get(\'candidates\'):\\n        lines.extend([\'\', \'원문 위치:\', \'\'])\\n        for item in audit[\'candidates\']:\\n            c = item[\'citations\'][0]\\n            locations = \' / \'.join(dict.fromkeys(span.get(\'locator\', \'\') for span in c.get(\'source_ranges\') or []))\\n            lines.append(\'- \' + escape(item[\'label\']) + \': \' + escape(c.get(\'file_name\', \'\')) + \' · \' + escape(locations or c.get(\'locator\', \'\')))\\n    lines.extend([\'\', \'추가 API 호출: 0회. 자동 의미 검증·사람 검토 완료 판정은 하지 않았습니다.\'])\\n    return \'\\\\n\'.join(lines)\\n", "rfp_phase5/summary.py": "\\"\\"\\"Source-validated full-input RFP summaries with eleven explicit output fields.\\n\\nThe Phase 4 planner and quote recovery remain the source of truth. This layer\\nadds field metadata and deterministic presentation, without an extra model\\nreduction call or an automatic assertion of semantic completeness.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport copy\\nimport hashlib\\nimport html\\nimport json\\nimport re\\n\\nfrom rfp_phase4.llm import LLMError\\nfrom rfp_phase4.summarization import FullDocumentSummarizer\\n\\nfrom .fields import FIELDS, FIELD_SCHEMA, FIELD_QUESTION, parse_field_claim, parse_missing_label\\n\\n\\nTASK = \'phase5_field_summary_map_v1\'\\n_STATE_LABELS = {\\n    \'evidence_found\': \'근거 추출됨 · 검토 필요\',\\n    \'notice_reference\': \'공고문 등 별도 자료 참조\',\\n    \'conflict_candidates\': \'상충 후보 · 확인 필요\',\\n    \'not_found_in_processed_input\': \'처리한 입력에서 미확인\',\\n    \'not_processed\': \'미실행·미처리\',\\n}\\n\\n\\ndef _fingerprint(value):\\n    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True,\\n        separators=(\',\', \':\'), allow_nan=False).encode(\'utf-8\')).hexdigest()\\n\\n\\nclass _FieldMapLLM:\\n    \\"\\"\\"Map identical planning, preflight and generation inputs to one task/schema.\\"\\"\\"\\n    def __init__(self, llm):\\n        self.base = llm\\n        self.config = llm.config\\n        self.received_chunk_ids = []\\n        self.field_contract_failures = []\\n\\n    def request_input_bytes(self, task, question, evidence, *, schema=None):\\n        return self.base.request_input_bytes(TASK, question, evidence, schema=FIELD_SCHEMA)\\n\\n    def preflight_requests(self, requests):\\n        mapped = [{**request, \'task\': TASK, \'schema\': FIELD_SCHEMA} for request in requests]\\n        return self.base.preflight_requests(mapped)\\n\\n    def usage_report(self):\\n        return self.base.usage_report()\\n\\n    def generate(self, task, question, evidence, *, schema=None):\\n        response = self.base.generate(TASK, question, evidence, schema=FIELD_SCHEMA)\\n        self.received_chunk_ids.extend(chunk[\'chunk_id\'] for chunk in evidence)\\n        payload = response.get(\'payload\', {})\\n        try:\\n            for claim in payload.get(\'claims\', []):\\n                parse_field_claim(claim)\\n        except (ValueError, TypeError, AttributeError):\\n            # The base client has already counted any paid attempt/cache hit. Do\\n            # not retry here or pass malformed claims to the accepted batch list.\\n            self.field_contract_failures.append({\\n                \'chunk_ids\': [chunk[\'chunk_id\'] for chunk in evidence],\\n                \'reason\': \'field_marker_or_category_invalid\',\\n            })\\n            raise LLMError(\'Phase 5 field marker/category validation failed\') from None\\n        return response\\n\\n\\nclass _FieldInputSummarizer(FullDocumentSummarizer):\\n    def plan(self, doc_id, *, question=FIELD_QUESTION):\\n        plan = super().plan(doc_id, question=question)\\n        plan[\'base_plan_fingerprint\'] = plan[\'plan_fingerprint\']\\n        plan[\'field_schema_fingerprint\'] = _fingerprint(FIELD_SCHEMA)\\n        plan[\'task\'] = TASK\\n        plan[\'field_ids\'] = [field[\'id\'] for field in FIELDS]\\n        plan[\'plan_fingerprint\'] = _fingerprint({\\n            \'base\': plan[\'base_plan_fingerprint\'], \'task\': TASK,\\n            \'schema\': plan[\'field_schema_fingerprint\'],\\n        })\\n        fresh_blocks = []\\n        calls_limit = plan.get(\'llm_max_calls\')\\n        input_limit = plan.get(\'llm_max_total_input_bytes\')\\n        if not isinstance(calls_limit, int) or isinstance(calls_limit, bool):\\n            fresh_blocks.append(\'llm_max_calls_unavailable\')\\n        elif plan[\'map_calls\'] > calls_limit:\\n            fresh_blocks.append(\'max_calls_exceeded\')\\n        if not isinstance(input_limit, int) or isinstance(input_limit, bool):\\n            fresh_blocks.append(\'llm_max_total_input_bytes_unavailable\')\\n        elif plan[\'total_request_bytes\'] > input_limit:\\n            fresh_blocks.append(\'max_total_input_bytes_exceeded\')\\n        plan[\'fresh_session_uncached_within_caps\'] = not fresh_blocks\\n        plan[\'fresh_session_uncached_block_reasons\'] = fresh_blocks\\n        plan[\'fresh_session_uncached_scope\'] = (\\n            \'Fresh session, every request uncached. This informational flag does not change \'\\n            \'can_execute (batch/map limits). Execution preflight checks actual remaining \'\\n            \'session caps and exact-request cache before any API call.\'\\n        )\\n        return plan\\n\\n\\nclass FieldSummarizer:\\n    \\"\\"\\"A bounded full-document map pass followed by an exact eleven-row view.\\"\\"\\"\\n    def __init__(self, index, llm, registry, config=None):\\n        self.index, self.llm, self.registry = index, llm, registry\\n        supplied = dict(config or {})\\n        supplied.setdefault(\'requested_categories\', list(dict.fromkeys(field[\'category\'] for field in FIELDS)))\\n        self._adapter = _FieldMapLLM(llm)\\n        self._inner = _FieldInputSummarizer(index, self._adapter, registry, supplied)\\n\\n    def plan(self, doc_id):\\n        return self._inner.plan(doc_id, question=FIELD_QUESTION)\\n\\n    def summarize(self, doc_id):\\n        self._adapter.received_chunk_ids = []\\n        self._adapter.field_contract_failures = []\\n        result = self._inner.summarize(doc_id, question=FIELD_QUESTION)\\n        # A malformed marker is a received response, while still never a\\n        # source-validated processed batch. Keep the two coverage meanings apart.\\n        received = set(result.get(\'response_received_chunk_ids\', [])) | set(self._adapter.received_chunk_ids)\\n        result[\'response_received_chunk_ids\'] = [cid for cid in result[\'source_chunk_ids\'] if cid in received]\\n        if result[\'source_chunk_ids\']:\\n            result[\'input_chunk_coverage\'] = len(result[\'response_received_chunk_ids\']) / len(result[\'source_chunk_ids\'])\\n        result[\'field_contract_failures\'] = copy.deepcopy(self._adapter.field_contract_failures)\\n        result = self._field_result(result)\\n        if result.get(\'processed_chunk_ids\'):\\n            from .source_audit import audit_summary_sources, render_source_audit\\n            audit = audit_summary_sources(self.index, self.registry, result)\\n            result[\'source_audit\'] = audit\\n            if audit[\'candidates\']:\\n                labels = list(dict.fromkeys(item[\'label\'] for item in audit[\'candidates\']))\\n                result[\'missing_information\'].extend(\'원문 추가 확인 필요: \' + label for label in labels)\\n                if result.get(\'claims\'):\\n                    result[\'status\'] = \'partial\'\\n                result[\'markdown\'] += \'\\\\n\\\\n\' + render_source_audit(audit)\\n        return result\\n\\n    def _field_result(self, result):\\n        # Accepted claims are still the original source-validated payload texts.\\n        # The display view strips only the transport prefix, after validation.\\n        rows = {field[\'id\']: {\'field_id\': field[\'id\'], \'label\': field[\'label\'],\\n                \'state\': \'not_processed\', \'claims\': [], \'notes\': []} for field in FIELDS}\\n        for claim in result.get(\'claims\', []):\\n            parsed = parse_field_claim(claim)\\n            rows[parsed[\'field_id\']][\'claims\'].append({\\n                \'text\': parsed[\'text\'], \'mode\': parsed[\'mode\'],\\n                \'citations\': copy.deepcopy(claim[\'citations\']),\\n            })\\n        global_notes = []\\n        for batch in result.get(\'batch_outcomes\', []):\\n            if not batch.get(\'valid\'):\\n                continue\\n            for label in batch.get(\'batch_missing_information\', []):\\n                parsed = parse_missing_label(label)\\n                scoped = f\'부분 입력 {batch[\\"batch_id\\"]}에서 미확인: \'\\n                if parsed:\\n                    rows[parsed[\'field_id\']][\'notes\'].append(scoped + parsed[\'text\'])\\n                else:\\n                    global_notes.append(scoped + str(label))\\n        any_processed = bool(result.get(\'processed_chunk_ids\'))\\n        for row in rows.values():\\n            modes = {claim[\'mode\'] for claim in row[\'claims\']}\\n            if \'conflict_candidate\' in modes:\\n                row[\'state\'] = \'conflict_candidates\'\\n                row[\'notes\'].append(\'모델이 구분한 상충 후보입니다. 원문·적용 범위를 대조하기 전에는 충돌로 확정하지 않습니다.\')\\n            elif \'fact\' in modes:\\n                row[\'state\'] = \'evidence_found\'\\n            elif \'notice_reference\' in modes:\\n                row[\'state\'] = \'notice_reference\'\\n            else:\\n                row[\'state\'] = \'not_found_in_processed_input\' if any_processed else \'not_processed\'\\n            if \'notice_reference\' in modes:\\n                row[\'notes\'].append(\'별도 공고문 등의 실제 내용은 이번 RFP 입력만으로 확인하지 않았습니다.\')\\n            if len({claim[\'text\'] for claim in row[\'claims\']}) > 1:\\n                row[\'notes\'].append(\'서로 다른 진술을 모두 보존했습니다. 세부 내역·적용 범위가 다른지 검토하세요.\')\\n            row[\'notes\'] = list(dict.fromkeys(row[\'notes\']))\\n        result[\'fields\'] = list(rows.values())\\n        result[\'field_ids\'] = list(rows)\\n        result[\'task\'] = TASK\\n        result[\'field_schema_fingerprint\'] = _fingerprint(FIELD_SCHEMA)\\n        result[\'semantic_quality_verified\'] = False\\n        result[\'automatic_conflict_resolution\'] = False\\n        result[\'field_uncertainty_notes\'] = list(dict.fromkeys(global_notes))\\n        result[\'phase4_missing_information\'] = result.get(\'missing_information\', [])\\n        missing = []\\n        for row in rows.values():\\n            if row[\'state\'] != \'evidence_found\':\\n                missing.append(f\'{row[\\"label\\"]}: {_STATE_LABELS[row[\\"state\\"]]}\')\\n            missing.extend(f\'{row[\\"label\\"]}: {note}\' for note in row[\'notes\'])\\n        missing.extend(global_notes)\\n        if result.get(\'execution_status\') == \'incomplete\':\\n            missing.append(\'일부 원문 청크를 검증된 요약으로 처리하지 못했습니다. 전체 문서 요약으로 확정할 수 없습니다.\')\\n        result[\'missing_information\'] = list(dict.fromkeys(missing))\\n        if result.get(\'claims\'):\\n            # Complete processing is necessary, but never called semantic proof.\\n            result[\'status\'] = (\'answered\' if result.get(\'execution_status\') == \'completed\'\\n                                and not result[\'missing_information\'] else \'partial\')\\n        result[\'markdown\'] = render_field_summary(result)\\n        return result\\n\\n\\ndef _escape(value):\\n    text = html.escape(str(value), quote=False)\\n    text = re.sub(r\'([\\\\\\\\`*_{}\\\\[\\\\]()#!+\\\\-.])\', r\'\\\\\\\\\\\\1\', text)\\n    return text.replace(\'|\', \'&#124;\').replace(\'\\\\r\', \'\').replace(\'\\\\n\', \' / \')\\n\\n\\ndef render_field_summary(result):\\n    \\"\\"\\"Render accepted field data; no unchecked model prose is interpolated.\\"\\"\\"\\n    lines = [\'| 항목 | 내용 | 확인 상태 |\', \'| --- | --- | --- |\']\\n    citations, by_key = [], {}\\n    for row in result.get(\'fields\', []):\\n        items = []\\n        for claim in row[\'claims\']:\\n            numbers = []\\n            for citation in claim[\'citations\']:\\n                key = (citation.get(\'doc_id\'), citation[\'chunk_id\'], citation[\'quote\'])\\n                if key not in by_key:\\n                    by_key[key] = len(citations) + 1\\n                    citations.append(citation)\\n                numbers.append(f\'[{by_key[key]}]\')\\n            items.append(_escape(claim[\'text\']) + \' \' + \'\'.join(numbers))\\n        text = \' / \'.join(items) if items else \'이번에 처리한 입력에서 근거를 확보하지 못했습니다.\'\\n        if row[\'state\'] == \'not_processed\':\\n            text = \'아직 요약을 실행하지 않았거나 이 항목을 처리하지 못했습니다.\'\\n        lines.append(f\'| {_escape(row[\\"label\\"])} | {text} | {_escape(_STATE_LABELS[row[\\"state\\"]])} |\')\\n    if citations:\\n        lines.extend([\'\', \'원문 근거:\', \'\'])\\n        for number, citation in enumerate(citations, 1):\\n            lines.append(f\'{number}. {_escape(citation.get(\\"file_name\\", \\"\\"))} · \'\\n                         f\'{_escape(citation.get(\\"locator\\", \\"\\"))} — “{_escape(citation[\\"quote\\"])}”\')\\n    notes = [f\'{row[\\"label\\"]}: {note}\' for row in result.get(\'fields\', []) for note in row[\'notes\']]\\n    notes.extend(result.get(\'field_uncertainty_notes\', []))\\n    if result.get(\'execution_status\') not in {\'completed\', \'incomplete\'}:\\n        notes.insert(0, \'요약을 실행하지 못했습니다. 실행 상태와 전체 입력 계획을 확인하세요.\')\\n    elif result.get(\'execution_status\') == \'incomplete\':\\n        notes.insert(0, \'일부 입력이 미처리 상태입니다. 이 결과를 전체 문서 요약으로 확정할 수 없습니다.\')\\n    if notes:\\n        lines.extend([\'\', \'추가 확인 사항:\', \'\'])\\n        lines.extend(\'- \' + _escape(note) for note in dict.fromkeys(notes))\\n    lines.extend([\'\', \'입력 처리율과 인용문 출처를 검사했습니다. 항목의 정확성·조건 보존·요약 완전성은 사람이 원문과 대조해야 합니다.\'])\\n    return \'\\\\n\'.join(lines)\\n", "tests/test_phase5_source_audit.py": "\\"\\"\\"Limited omission hints must remain source-bound and separate from answers.\\"\\"\\"\\nimport copy\\nimport unittest\\n\\nfrom rfp_phase4.evidence import EvidenceRegistry\\nfrom rfp_phase5.source_audit import audit_summary_sources, render_source_audit\\nfrom rfp_phase5.summary import FieldSummarizer\\nimport test_phase5_summary as fixture\\n\\n\\ndef source_case():\\n    docs, chunks = fixture.corpus(count=1)\\n    text = (\'사업예산 : 12,345,000원(부가세 제외)\\\\n\'\\n            \'본 사업은 단가계약으로, 대금은 실제 대상인원에 대하여 정산하여 지급\\\\n\'\\n            \'기간 : 계약체결일 ~ 2030. 4. 1\\\\n\'\\n            \'후속절차 지원은 과업기간 이후에도 포함\\\\n\'\\n            \'낙찰자 선정방식 : 적격심사(제한적최저가) 낙찰제\')\\n    docs[0][\'blocks\'][0][\'text\'] = text\\n    chunks[0].update(text=text, text_bytes=len(text.encode()))\\n    span = chunks[0][\'source_spans\'][0]\\n    span.update(char_end=len(text), chunk_end=len(text))\\n    claims = [{\'doc_id\': \'d1\', \'category\': \'budget\', \'text\': \'[budget_vat|fact] 12,345,000원(부가세 제외)\',\\n               \'citations\': [{\'chunk_id\': \'c0\', \'quote\': text.splitlines()[0]}]}]\\n    result = {\'doc_id\': \'d1\', \'status\': \'partial\', \'execution_status\': \'completed\',\\n              \'claims\': claims, \'missing_information\': [\'기타 미확인\'],\\n              \'source_chunk_ids\': [\'c0\'], \'processed_chunk_ids\': [\'c0\']}\\n    return fixture.Index(chunks), EvidenceRegistry(docs, chunks), result\\n\\n\\nclass SourceAuditTests(unittest.TestCase):\\n    def test_three_hints_use_processed_original_source_without_mutating_answer(self):\\n        index, registry, result = source_case()\\n        before = copy.deepcopy(result)\\n        audit = audit_summary_sources(index, registry, result)\\n        self.assertEqual(audit[\'candidate_count\'], 3)\\n        self.assertEqual({c[\'field_id\'] for c in audit[\'candidates\']}, {\'budget_vat\', \'duration\', \'evaluation\'})\\n        self.assertTrue(audit[\'original_quotes_revalidated\'])\\n        self.assertEqual(audit[\'api_calls\'], 0)\\n        self.assertFalse(audit[\'semantic_completeness_verified\'])\\n        self.assertFalse(audit[\'human_review_completed\'])\\n        self.assertEqual(result, before)\\n        self.assertTrue(all(c[\'citations\'][0][\'provenance_verified\'] for c in audit[\'candidates\']))\\n        self.assertTrue(all(not c[\'automatically_added_to_answer\'] for c in audit[\'candidates\']))\\n\\n    def test_unprocessed_chunks_are_never_scanned(self):\\n        index, registry, result = source_case()\\n        result.update(claims=[], processed_chunk_ids=[], status=\'error\')\\n        audit = audit_summary_sources(index, registry, result)\\n        self.assertEqual(audit[\'candidate_count\'], 0)\\n        self.assertFalse(audit[\'all_input_chunks_processed\'])\\n\\n    def test_unknown_scope_and_duplicate_source_ids_are_rejected(self):\\n        for mode in (\'doc\', \'processed\', \'source\'):\\n            index, registry, result = source_case()\\n            if mode == \'doc\': result[\'doc_id\'] = \'other\'\\n            if mode == \'processed\': result[\'processed_chunk_ids\'] *= 2\\n            if mode == \'source\': result[\'source_chunk_ids\'] = []\\n            with self.assertRaises(ValueError): audit_summary_sources(index, registry, result)\\n\\n    def test_fabricated_existing_quote_is_not_accepted_by_source_audit(self):\\n        index, registry, result = source_case()\\n        result[\'claims\'][0][\'citations\'][0][\'quote\'] = \'원문에 없는 금액 999원\'\\n        with self.assertRaises(ValueError): audit_summary_sources(index, registry, result)\\n\\n    def test_same_condition_in_another_field_remains_a_field_review_hint(self):\\n        index, registry, result = source_case()\\n        result[\'claims\'][0][\'text\'] = \'[budget_vat|fact] 후속지원은 과업기간 이후에도 포함\'\\n        audit = audit_summary_sources(index, registry, result)\\n        self.assertIn(\'duration\', [c[\'field_id\'] for c in audit[\'candidates\']])\\n\\n    def test_presence_check_is_limited_and_does_not_infer_scores(self):\\n        index, registry, result = source_case()\\n        result[\'claims\'].append({\'doc_id\': \'d1\', \'category\': \'evaluation\',\\n            \'text\': \'[evaluation|fact] 적격심사 방식이다. 배점은 미확인이다.\',\\n            \'citations\': [{\'chunk_id\': \'c0\', \'quote\': index.chunks[0][\'text\'].splitlines()[-1]}]})\\n        audit = audit_summary_sources(index, registry, result)\\n        self.assertNotIn(\'evaluation\', [c[\'field_id\'] for c in audit[\'candidates\']])\\n        self.assertIsNone(audit[\'semantic_accuracy\'])\\n\\n    def test_selection_heading_is_not_an_operative_method_value(self):\\n        import re\\n        from rfp_phase5.source_audit import RULES\\n        pattern = next(rule[2] for rule in RULES if rule[0] == \'selection_method\')\\n        self.assertIsNone(re.search(pattern, \'3. 사업예산 및 낙찰자 선정방식\\\\t1\'))\\n        self.assertIsNone(re.search(pattern, \'낙찰자 선정방식\'))\\n        self.assertIsNotNone(re.search(pattern, \'낙찰자 선정방식 : 적격심사\'))\\n\\n    def test_renderer_escapes_links_images_and_table_delimiters(self):\\n        index, registry, result = source_case()\\n        audit = audit_summary_sources(index, registry, result)\\n        audit[\'candidates\'][0][\'source_quote\'] += \' ![unsafe](https://example.test) | <script>\'\\n        rendered = render_source_audit(audit)\\n        self.assertNotIn(\'![unsafe]\', rendered)\\n        self.assertIn(\'&#124;\', rendered)\\n        self.assertNotIn(\'<script>\', rendered)\\n\\n    def test_regular_summarizer_attaches_hints_without_extra_model_calls(self):\\n        index, registry, _ = source_case()\\n        llm = fixture.OfflineLLM(all_fields=True)\\n        result = FieldSummarizer(index, llm, registry).summarize(\'d1\')\\n        self.assertEqual(len(llm.calls), 1)\\n        self.assertEqual(len(result[\'claims\']), 11)\\n        self.assertEqual(result[\'status\'], \'partial\')\\n        # This fixture\'s budget citation already includes the unit-price line;\\n        # only the duration and selection-method candidates remain new hints.\\n        self.assertEqual({c[\'field_id\'] for c in result[\'source_audit\'][\'candidates\']}, {\'duration\', \'evaluation\'})\\n        self.assertIn(\'원문 추가 확인 필요:\', \'\\\\n\'.join(result[\'missing_information\']))\\n\\n\\nif __name__ == \'__main__\':\\n    unittest.main()\\n"}')
patch_report = apply_source_audit(ROOT, MANIFEST, EMBEDDED_FILES)
print(json.dumps(patch_report, ensure_ascii=False, indent=2))


{
  "patch_id": "phase5-source-audit-v1",
  "modified_files": [
    "rfp_phase5/source_audit.py",
    "rfp_phase5/summary.py",
    "tests/test_phase5_source_audit.py"
  ],
  "already_applied_files": [],
  "backup_directory": "/Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase5_local_20261111/phase5_outputs/patch_backups/20261111T114237_568995Z",
  "api_calls": 0,
  "data_or_index_modified": false,
  "cache_modified": false,
  "config_or_notebook_flags_modified": false,
  "retry_executed": false,
  "restart_08_notebook_kernel_before_future_generation": true,
  "original_answers_modified": false,
  "semantic_quality_verified": false
}


## 2. 기존 결과와 원문 대조

출처를 먼저 다시 검증한 뒤, 실제로 처리한 청크만 대조합니다. 원문을 못 읽은 범위는 검사했다고 표시하지 않습니다.
아래 표의 인용은 원문에 있는 내용이며, 의미 판단과 사용자 검토는 별도입니다.
다른 문서를 선택하면 이 사례와 다른 후보가 표시될 수 있습니다.


In [3]:
from datetime import datetime, timezone
import importlib
import uuid

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rfp_phase5.runtime import bootstrap_phase5
import rfp_phase5.source_audit as audit_module
audit_module = importlib.reload(audit_module)
if RESULT_PATH is None:
    paths = sorted((ROOT / 'phase5_outputs/runs').glob('*/result.json'),
                   key=lambda p: (p.stat().st_mtime_ns, str(p)))
    if not paths:
        raise FileNotFoundError('기존 result.json이 없습니다. RESULT_PATH에 경로를 지정하세요.')
    result_path = paths[-1]
else:
    result_path = Path(RESULT_PATH).expanduser()
    if not result_path.is_absolute():
        result_path = ROOT / result_path
saved_result = json.loads(result_path.read_text(encoding='utf-8'))
print('검토할 기존 결과:', result_path)
app = bootstrap_phase5(ROOT)
try:
    source_audit = audit_module.audit_summary_sources(app['index'], app['registry'], saved_result)
finally:
    app['index'].close()
review_markdown = audit_module.render_source_audit(source_audit)
try:
    from IPython.display import Markdown, display
    display(Markdown(review_markdown))
except ImportError:
    print(review_markdown)
folder = ROOT / 'phase5_outputs/source_reviews' / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ_') + uuid.uuid4().hex[:8])
folder.mkdir(parents=True, exist_ok=False)
(folder / 'source_audit.json').write_text(json.dumps(source_audit, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(folder / 'source_audit.md').write_text(review_markdown + '\n', encoding='utf-8')
print(json.dumps({k: v for k, v in source_audit.items() if k != 'candidates'}, ensure_ascii=False, indent=2))
print('원문 대조 기록:', folder)
print('원래 result.json은 그대로 보존했습니다. 추가 API 호출: 0회')


검토할 기존 결과: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase5_local_20261111/phase5_outputs/runs/20261111T111052_567782Z_c689d44b/result.json


원문 추가 확인

이미 전달된 원문에서 찾은 내용입니다. 기존 생성 답변에 자동으로 추가하지 않았습니다.

| 항목 | 원문에서 찾은 내용 | 확인할 점 |
| --- | --- | --- |
| 예산·VAT | ※ 본 사업은 단가계약으로, 대금은 실제 대상인원에 대하여 정산하여 지급 | 단가계약과 실제 대상·물량에 따른 정산 조건을 확인하세요\. |
| 수행기간 | ※ 당해 연도의 부당 공제자 통보 시, 후속절차 지원은 과업기간 이후에도 포함 | 계약상 기간이 끝난 뒤에도 남는 지원 의무가 있는지 확인하세요\. |
| 평가기준 | 2\) 낙찰자 선정방식 : 적격심사\(제한적최저가\) 낙찰제 | 선정방식과 세부 평가항목·배점을 구분하세요\. 선정방식만으로 배점표를 추정하지 않습니다\. |

원문 위치:

- 예산·VAT: 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 75
- 수행기간: 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 62
- 평가기준: 인천공항시설관리\(주\)\_2026\_2026년도 귀속 연말정산 위탁운영 용역\_과업내용서\.hwp · BodyText/Section0 / paragraph 74

추가 API 호출: 0회. 자동 의미 검증·사람 검토 완료 판정은 하지 않았습니다.

{
  "version": "phase5-source-audit-v1",
  "doc_id": "1Eq8gIdUnnLRbbEBiaU29wYji2YglNNuu",
  "source_sha256": "7a592ea538ed00d8dca697c8cab8fadd110ec2a15902f8235b80bb7d960027bc",
  "input_result_fingerprint": "5987de6634acab6ac30d505cddcf072e05f1559a712621aff6ac2f959cb3700a",
  "processed_chunk_count": 18,
  "document_chunk_count": 18,
  "all_input_chunks_processed": true,
  "original_claim_count": 6,
  "original_citation_count": 20,
  "original_quotes_revalidated": true,
  "candidate_count": 3,
  "semantic_accuracy": null,
  "semantic_completeness_verified": false,
  "human_review_completed": false,
  "api_calls": 0,
  "scope": "이미 처리된 원문의 일부 표면 규칙을 대조한 추가 확인 후보. 모든 누락 탐지나 의미적 정답 판정은 아닙니다."
}
원문 대조 기록: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase5_local_20261111/phase5_outputs/source_reviews/20261111T114243_631706Z_10f61862
원래 result.json은 그대로 보존했습니다. 추가 API 호출: 0회
